# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hardik-c123/FlyRank-AI-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
**My data contract: five answers**

1. **What one row means for my lane:** one content item (`content_hash_id`) inside one client (`client_hash_id`), summarised over a feature window. The raw table is finer: one row per day x client x content item.
2. **Tables I will use:** `fact_content_daily_performance` (main source) and `dim_clients` (to check how much history each client has).
3. **Time window:** features come from one mid-panel month, `month=2026-03`. The outcome comes from the next month, `month=2026-04`. The final month (June 2026) is a sealed test month and I do not use it, and I do not use the `_sample` table to build label logic.
4. **What I would predict or rank (proxy label):** `declined_apr` = 1 when a page's April impressions are below 80% of its March impressions, among pages with at least 100 March impressions. This is a proxy, not proof of a lasting decline. I would use the predicted probability to rank pages for human review (Lane 2).
5. **One thing I deliberately exclude:** every April metric as a feature, because April is the outcome window and using it would leak the answer.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
contract = {
    "row": "one content item (content_hash_id) within one client (client_hash_id)",
    "raw_grain": "report_date x client_hash_id x content_hash_id",
    "tables": ["fact_content_daily_performance", "dim_clients"],
    "feature_window": "month=2026-03",
    "outcome_window": "month=2026-04",
    "label": "declined_apr = April impressions < 0.8 x March impressions (March impressions >= 100)",
    "excluded": "all April metrics as features (leakage)",
}
for k, v in contract.items():
    print(f"{k}: {v}")

row: one content item (content_hash_id) within one client (client_hash_id)
raw_grain: report_date x client_hash_id x content_hash_id
tables: ['fact_content_daily_performance', 'dim_clients']
feature_window: month=2026-03
outcome_window: month=2026-04
label: declined_apr = April impressions < 0.8 x March impressions (March impressions >= 100)
excluded: all April metrics as features (leakage)


## 2. Fields: feature / label / context / excluded



*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `impressions_mar`, `ctr_mar`, `avg_position_mar`, `days_with_impressions_mar`, `sessions_mar` | Observable signals built only from March rows, so they are known at the decision moment (end of March). |
| **Label** | `declined_apr` (built from `impressions_apr`) | Future observed outcome, measured after the decision moment. |
| **Context** | `client_hash_id`, `content_hash_id`, `report_date`, `ga4_data_available` | Used for joins, grouping, client-holdout validation and checking tracking coverage. Not fed to the model as meaning. |
| **Excluded** | April metrics as features | They are the outcome window, so they would leak the answer. |
| **Excluded** | `sessions_ai` | Very sparse (about 30k rows out of 78.8M), so it would look useful and mean nothing. |
| **Excluded** | `_sample` table and the June 2026 month | The last month is the natural outcome window and is kept sealed as a test month. |

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
buckets = {
    "feature": ["impressions_mar", "ctr_mar", "avg_position_mar",
                "days_with_impressions_mar", "sessions_mar"],
    "label": ["declined_apr"],
    "context": ["client_hash_id", "content_hash_id", "report_date", "ga4_data_available"],
    "excluded": {
        "April metrics as features": "outcome window, would leak the answer",
        "sessions_ai": "too sparse to support a model",
        "_sample table / June 2026": "final month kept sealed for testing",
    },
}
for k, v in buckets.items():
    print(k, "->", v)

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*
Three verification queries on `month=2026-03`: (1) grain, (2) date span of my slice, (3) availability filtered with `IS TRUE`. Then a five-feature frame, and the leakage trap.

**Five features and when each is available:**

| Feature | Available when? |
|---|---|
| `impressions_mar` | Knowable at the end of March: a sum of March daily impressions, nothing after 31 March. |
| `ctr_mar` | Knowable at the end of March: March clicks divided by March impressions. |
| `avg_position_mar` | Knowable at the end of March: the average of March daily positions. |
| `days_with_impressions_mar` | Knowable at the end of March: the count of March days that had impressions. |
| `sessions_mar` | Knowable at the end of March: March GA4 sessions, only from days where GA4 data was available. |

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip install -q duckdb
import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")  # never print this
con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"
MAR = f"read_parquet('{REL}/month=2026-03/**/*.parquet')"
APR = f"read_parquet('{REL}/month=2026-04/**/*.parquet')"

# Query 1: grain. One row should be one report_date x client x content item.
q1 = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           (SELECT COUNT(*) FROM (
               SELECT DISTINCT report_date, client_hash_id, content_hash_id FROM {MAR}
           )) AS n_distinct_keys
    FROM {MAR}
""").df()
q1["duplicate_rows"] = q1["n_rows"] - q1["n_distinct_keys"]
print("QUERY 1 - grain")
print(q1.to_string(index=False), "\n")

# Query 2: date span and size of my slice.
q2 = con.sql(f"""
    SELECT MIN(report_date) AS first_day, MAX(report_date) AS last_day,
           COUNT(DISTINCT report_date) AS n_days,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM {MAR}
""").df()
print("QUERY 2 - date span")
print(q2.to_string(index=False), "\n")

# Query 3: availability. Keep only rows where GA4 data exists (IS TRUE).
q3 = con.sql(f"""
    SELECT (SELECT COUNT(*) FROM {MAR}) AS all_rows,
           (SELECT COUNT(*) FROM {MAR} WHERE ga4_data_available IS TRUE) AS rows_ga4_true
""").df()
q3["share_surviving"] = (q3["rows_ga4_true"] / q3["all_rows"]).round(3)
print("QUERY 3 - availability (ga4_data_available IS TRUE)")
print(q3.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

QUERY 1 - grain
 n_rows  n_distinct_keys  duplicate_rows
9841378          9841378               0 

QUERY 2 - date span
 first_day   last_day  n_days  n_clients  n_content_items
2026-03-01 2026-03-31      31         55           331437 



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

QUERY 3 - availability (ga4_data_available IS TRUE)
 all_rows  rows_ga4_true  share_surviving
  9841378         413966            0.042


In [4]:
# Find the real column names in the table (names for metrics are not guaranteed)
cols = con.sql(f"DESCRIBE SELECT * FROM {MAR}").df()["column_name"].tolist()

def pick(candidates, must, avoid=()):
    for c in candidates:
        if c in cols:
            return c
    for c in cols:
        low = c.lower()
        if must in low and not any(a in low for a in avoid):
            return c
    raise KeyError(f"No column for '{must}'. Columns are: {cols}")

IMP = pick(["gsc_impressions", "impressions"], "impress")
CLK = pick(["gsc_clicks", "clicks"], "click")
POS = pick(["gsc_avg_position", "avg_position", "gsc_position", "position"], "position")
SES = pick(["ga4_sessions", "sessions"], "session", avoid=("_ai",))
print("Using columns:", {"impressions": IMP, "clicks": CLK, "position": POS, "sessions": SES})

# Decision moment = end of March. Features use March rows only.
feat = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM({IMP})                                              AS impressions_mar,
           SUM({CLK}) / NULLIF(SUM({IMP}), 0)                      AS ctr_mar,
           AVG({POS})                                              AS avg_position_mar,
           COUNT(*) FILTER (WHERE {IMP} > 0)                       AS days_with_impressions_mar,
           SUM({SES}) FILTER (WHERE ga4_data_available IS TRUE)    AS sessions_mar
    FROM {MAR}
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM({IMP}) >= 100
""").df()

# Outcome month (April) is used ONLY to build the label.
apr = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM({IMP}) AS impressions_apr
    FROM {APR}
    GROUP BY client_hash_id, content_hash_id
""").df()

df = feat.merge(apr, on=["client_hash_id", "content_hash_id"], how="left")
df["impressions_apr"] = df["impressions_apr"].fillna(0)   # assumption: no April row = 0 impressions
df["declined_apr"] = (df["impressions_apr"] < 0.8 * df["impressions_mar"]).astype(int)

FEATURES = ["impressions_mar", "ctr_mar", "avg_position_mar",
            "days_with_impressions_mar", "sessions_mar"]
print("Rows:", len(df), "| Clients:", df["client_id"].nunique() if "client_id" in df else df["client_hash_id"].nunique())
print("Share labelled declined:", round(df["declined_apr"].mean(), 3))
df[["client_hash_id", "content_hash_id"] + FEATURES + ["declined_apr"]].head(10)

Using columns: {'impressions': 'gsc_impressions', 'clicks': 'gsc_clicks', 'position': 'gsc_avg_position', 'sessions': 'ga4_sessions'}


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 101441 | Clients: 44
Share labelled declined: 0.517


,client_hash_id,content_hash_id,impressions_mar,ctr_mar,avg_position_mar,days_with_impressions_mar,sessions_mar,declined_apr
0,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,602.0,0.006645,4.428747,29,NaN,0
1,client_62f4a7e64f5e0096,content_275b6f7f733016d4,810.0,0.001235,4.866123,29,NaN,1
2,client_62f4a7e64f5e0096,content_755d951187fcd70a,1858.0,0.003229,1.854929,30,NaN,0
3,client_62f4a7e64f5e0096,content_92c381fbd361212e,536.0,0.001866,4.442543,29,NaN,1
4,client_62f4a7e64f5e0096,content_97188a7032a705cf,496.0,0.006048,4.018509,29,NaN,1
5,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,0.002028,8.240351,31,NaN,0
6,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,0.001418,5.956862,31,NaN,1
7,client_62f4a7e64f5e0096,content_db118320918083bc,245.0,0.000000,6.964106,30,NaN,1
8,client_62f4a7e64f5e0096,content_df22bda1218f13ff,2099.0,0.000476,3.066796,31,NaN,1
9,client_62f4a7e64f5e0096,content_0e1235786b36b7e3,735.0,0.021769,3.173884,29,NaN,0


In [5]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

y = df["declined_apr"]
groups = df["client_hash_id"]
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
                           .split(df, y, groups=groups))

def quick_score(feature_cols):
    model = HistGradientBoostingClassifier(random_state=42)
    model.fit(df.iloc[train_idx][feature_cols], y.iloc[train_idx])
    proba = model.predict_proba(df.iloc[test_idx][feature_cols])[:, 1]
    return roc_auc_score(y.iloc[test_idx], proba)

# Honest score, five features only (client-holdout split)
honest_before = quick_score(FEATURES)

# THE TRAP: add ONE label-derived column on purpose (April impressions)
df["leak_impressions_apr"] = df["impressions_apr"]
leaky = quick_score(FEATURES + ["leak_impressions_apr"])

# Remove it and report the honest number again
df = df.drop(columns=["leak_impressions_apr"])
honest_after = quick_score(FEATURES)

print(f"Honest AUC (5 features):          {honest_before:.3f}")
print(f"AUC with the leaked column added: {leaky:.3f}   <- looks 'too good'")
print(f"Honest AUC after deleting leak:   {honest_after:.3f}")

Honest AUC (5 features):          0.665
AUC with the leaked column added: 1.000   <- looks 'too good'
Honest AUC after deleting leak:   0.665


**The trap (leakage lesson).** I added one label-derived column, April impressions, as a feature. The quick score (ROC AUC) jumped from **0.666** to **1.000**, which is close to perfect. That is a warning sign, not a success: April impressions are the outcome window, so the model was just reading the answer. I deleted the column and the honest score went back to **0.666**. This is the number I trust, and it is directional, not a guarantee.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
**Named limitation (main one): unbalanced history and GSC-only early rows.** Different clients start tracking at different dates, and rows from before a client's GA4 start contain search data only (`ga4_data_available = FALSE`). So a missing or zero `sessions_mar` can mean "no tracking yet", not "no traffic". I keep `sessions_mar` as missing for those rows and do not read it as zero.

**Other limits I know about:**
- My label compares March with April, so a drop can also come from seasonality, consolidation into another page, or noise, not only a real decline.
- A page with no April row is treated as zero April impressions. That is an assumption, and it can be wrong for a client whose data stops early.
- One month of features is a small window. Results here are observational and directional, and have to be re-checked on other months.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# How much of my March slice has GA4 tracking, per client?
cov = con.sql(f"""
    SELECT client_hash_id,
           COUNT(*) AS rows_mar,
           ROUND(AVG(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END), 3) AS share_ga4_available
    FROM {MAR}
    GROUP BY client_hash_id
""").df()

print("Clients in March slice:", len(cov))
print("Clients with NO GA4 rows in March:", int((cov["share_ga4_available"] == 0).sum()))
print("Clients with PARTIAL GA4 coverage:",
      int(((cov["share_ga4_available"] > 0) & (cov["share_ga4_available"] < 1)).sum()))
cov.sort_values("share_ga4_available").head(10)

Clients in March slice: 55
Clients with NO GA4 rows in March: 16
Clients with PARTIAL GA4 coverage: 39


,client_hash_id,rows_mar,share_ga4_available
0,client_8dbf3abdf07569e0,17825,0.0
1,client_625b6439094e23e4,988497,0.0
14,client_80ee5b7bd5f4eb89,6076,0.0
13,client_770e8e5faa9cddfe,6200,0.0
12,client_599043c0ff13edea,22537,0.0
29,client_0797ff3a1fc9a6a5,8060,0.0
18,client_795153d5b7850ccf,78709,0.0
17,client_2b4306c3ed003f01,375906,0.0
24,client_a1203ffecad62470,2325,0.0
52,client_400c21c81c8b46ef,106808,0.0


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.